# 31_gold_fact — Gold Fact Table (CDF-incremental)

Loads `gold.fact_listening` incrementally from `silver.events` Change Data Feed.

**Flow:**
1. Read `ops.pipeline_state` for `last_version` of `silver.events`
2. Bootstrap: if `last_version IS NULL`, full snapshot load from Silver
3. Skip if `last_version == current_version`
4. Else: read CDF changes, enrich with surrogate keys, MERGE into fact
5. Propagate deletes from Silver to fact
6. Update `ops.pipeline_state.last_version` after successful commit

In [ ]:
%run ./00_config


## Step 1: Read pipeline state and determine CDF range

In [ ]:
from datetime import datetime, timezone
from pyspark.sql import Window

_t0 = datetime.now(timezone.utc)

# Current version of silver.events
cur_ver = spark.sql(
    f'DESCRIBE HISTORY {table_ref(SILVER_EVENTS)} LIMIT 1'
).first()['version']

state = spark.sql(
    f"SELECT last_version, cdf_enabled_at_version "
    f"FROM {table_ref(PIPELINE_STATE)} "
    f"WHERE table_name='{SILVER_EVENTS}'"
).first()

last_ver = state['last_version'] if state else None
cdf_enabled_at = state['cdf_enabled_at_version'] if state else 0

print(f'silver.events: current version={cur_ver}, last_version={last_ver}, '
      f'cdf_enabled_at={cdf_enabled_at}')


## Step 2: Load or skip

In [ ]:
# Skip if already up to date
if last_ver is not None and last_ver == cur_ver:
    log_run('SILVER_TO_GOLD','gold.fact_listening',
            f'cdf v{last_ver}..v{cur_ver}',_t0,'SKIPPED_UNCHANGED',rows_read=0)
    print('fact_listening: no new Silver versions — SKIPPED_UNCHANGED')
    dbutils.notebook.exit('SKIPPED_UNCHANGED')


In [ ]:
# ---------------------------------------------------------------------------
# Determine source DataFrame: full snapshot (bootstrap) or CDF delta
# ---------------------------------------------------------------------------
is_bootstrap = (last_ver is None)

if is_bootstrap:
    print(f'BOOTSTRAP: loading full snapshot of silver.events (version {cur_ver})')
    # Full snapshot — treat all rows as inserts
    df_changes = spark.table(SILVER_EVENTS) \
        .withColumn('_change_type', F.lit('insert'))
    param_str = f'bootstrap v0..v{cur_ver}'
else:
    print(f'CDF INCREMENTAL: reading changes v{last_ver+1}..v{cur_ver}')
    df_cdf = spark.read.format('delta') \
        .option('readChangeFeed', 'true') \
        .option('startingVersion', last_ver + 1) \
        .table(SILVER_EVENTS)
    # Drop update_preimage rows (we only need update_postimage and delete)
    df_cdf = df_cdf.filter(F.col('_change_type') != 'update_preimage')
    # Keep only the latest change per event_id across this version range
    w = Window.partitionBy('event_id').orderBy(F.col('_commit_version').desc())
    df_changes = df_cdf.withColumn('_rn', F.row_number().over(w)) \
                       .filter(F.col('_rn') == 1).drop('_rn')
    param_str = f'cdf v{last_ver+1}..v{cur_ver}'

n_read = df_changes.count()
print(f'Changes to process: {n_read:,}')


## Step 3: Enrich changes with surrogate keys

In [ ]:
# ---------------------------------------------------------------------------
# Split: deletes vs inserts/updates
# ---------------------------------------------------------------------------
df_deletes = df_changes.filter(F.col('_change_type') == 'delete').select('event_id')
df_upserts = df_changes.filter(F.col('_change_type').isin('insert','update_postimage'))

n_deletes = df_deletes.count()
n_upserts = df_upserts.count()
print(f'  Upserts: {n_upserts:,}, Deletes: {n_deletes:,}')

# ---------------------------------------------------------------------------
# Enrich upserts: join to dims for surrogate keys
# ---------------------------------------------------------------------------
try:
    _ts = F.current_timestamp()

    # dim_device lookup
    df_dev = spark.table(GOLD['dim_device']).select('device_sk','device_type')
    # dim_track lookup
    df_trk = spark.table(GOLD['dim_track']).select('track_sk','track_id')
    # dim_user: point-in-time join (played_at BETWEEN valid_from AND valid_to)
    df_usr = spark.table(GOLD['dim_user']).filter(F.col('user_id') != 'user_unknown') \
        .select('user_sk','user_id','valid_from','valid_to','country')

    # Compute fact columns
    df_enriched = df_upserts \
        .withColumn('date_key',
            F.date_format(F.col('played_at'),'yyyyMMdd').cast('int')) \
        .withColumn('hour_of_day',
            F.hour(F.col('played_at')).cast('tinyint')) \
        .withColumn('minutes_played',
            (F.col('ms_played') / 60000.0)) \
        .withColumn('play_count', F.lit(1)) \
        .withColumn('is_skipped',
            F.when(F.col('skipped').isNull(), F.lit(None).cast('int'))
             .when(F.col('skipped') == True, F.lit(1))
             .otherwise(F.lit(0)).cast('int'))

    # Join device dim (left join, default to UNKNOWN_SK)
    df_enriched = df_enriched.join(
        df_dev.withColumnRenamed('device_sk','_dev_sk'), 'device_type', 'left'
    ).withColumn('device_sk',
        F.coalesce(F.col('_dev_sk'), F.lit(UNKNOWN_SK).cast('bigint'))
    ).drop('_dev_sk')

    # Join track dim (left join, default to UNKNOWN_SK)
    df_enriched = df_enriched.join(
        df_trk.withColumnRenamed('track_sk','_trk_sk'), 'track_id', 'left'
    ).withColumn('track_sk',
        F.coalesce(F.col('_trk_sk'), F.lit(UNKNOWN_SK).cast('bigint'))
    ).drop('_trk_sk')

    # Point-in-time join to dim_user (played_at between valid_from and valid_to)
    # Use broadcast for small dim_user
    df_enriched = df_enriched.alias('e').join(
        F.broadcast(df_usr).alias('u'),
        (F.col('e.user_id') == F.col('u.user_id')) &
        (F.col('e.played_at') >= F.col('u.valid_from')) &
        (F.col('e.played_at') <  F.col('u.valid_to')),
        'left'
    ).withColumn('user_sk',
        F.coalesce(F.col('u.user_sk'), F.lit(UNKNOWN_SK).cast('bigint'))
    ).withColumn('country_dim', F.col('u.country')) \
     .drop('u.user_id','u.valid_from','u.valid_to','u.country','u.user_sk')

    # Row hash for idempotent MERGE (skip unchanged rows)
    df_enriched = df_enriched.withColumn('_row_hash',
        F.md5(F.concat_ws('||',
            F.col('user_sk').cast('string'), F.col('track_sk').cast('string'),
            F.col('device_sk').cast('string'), F.col('date_key').cast('string'),
            F.col('ms_played').cast('string'), F.col('is_skipped').cast('string'),
            F.col('completion_rate').cast('string')
        ))
    ).withColumn('created_at',     _ts) \
     .withColumn('updated_at',     _ts) \
     .withColumn('load_timestamp', _ts)

    df_enriched.createOrReplaceTempView('_fact_upserts')
    print(f'Enriched {n_upserts:,} upsert rows')

except Exception as _e:
    log_run('SILVER_TO_GOLD','gold.fact_listening',param_str,_t0,'FAILURE',
            rows_read=n_read,error_message=str(_e))
    raise


## Step 4: MERGE into gold.fact_listening

In [ ]:
try:
    # Apply deletes first
    if n_deletes > 0:
        df_deletes.createOrReplaceTempView('_fact_deletes')
        spark.sql(f"""
            MERGE INTO {table_ref(GOLD['fact_listening'])} AS tgt
            USING _fact_deletes AS src ON tgt.event_id = src.event_id
            WHEN MATCHED THEN DELETE
        """)
        print(f'Deleted {n_deletes:,} rows from fact_listening')

    # Upsert enriched rows
    if n_upserts > 0:
        spark.sql(f"""
            MERGE INTO {table_ref(GOLD['fact_listening'])} AS tgt
            USING _fact_upserts AS src ON tgt.event_id = src.event_id
            WHEN MATCHED AND tgt._row_hash <> src._row_hash THEN
                UPDATE SET
                    tgt.user_sk=src.user_sk, tgt.track_sk=src.track_sk,
                    tgt.device_sk=src.device_sk, tgt.date_key=src.date_key,
                    tgt.hour_of_day=src.hour_of_day, tgt.session_id=src.session_id,
                    tgt.data_source=src.data_source, tgt.country=src.country_dim,
                    tgt.ms_played=src.ms_played, tgt.minutes_played=src.minutes_played,
                    tgt.play_count=src.play_count, tgt.is_skipped=src.is_skipped,
                    tgt.completion_rate=src.completion_rate,
                    tgt._row_hash=src._row_hash,
                    tgt.updated_at=src.updated_at,
                    tgt.load_timestamp=src.load_timestamp
            WHEN NOT MATCHED THEN
                INSERT (event_id,user_sk,track_sk,device_sk,date_key,hour_of_day,
                        session_id,data_source,country,ms_played,minutes_played,
                        play_count,is_skipped,completion_rate,
                        _row_hash,created_at,updated_at,load_timestamp)
                VALUES (src.event_id,src.user_sk,src.track_sk,src.device_sk,
                        src.date_key,src.hour_of_day,src.session_id,src.data_source,
                        src.country_dim,src.ms_played,src.minutes_played,
                        src.play_count,src.is_skipped,src.completion_rate,
                        src._row_hash,src.created_at,src.updated_at,src.load_timestamp)
        """)

    # !! Read execution_metrics BEFORE updating pipeline_state and BEFORE OPTIMIZE !!
    m = execution_metrics(GOLD['fact_listening'])

    # Update pipeline_state — only after successful MERGE commit
    spark.sql(f"""
        UPDATE {table_ref(PIPELINE_STATE)}
        SET last_version={cur_ver}, updated_at=CURRENT_TIMESTAMP(),
            load_timestamp=CURRENT_TIMESTAMP()
        WHERE table_name='{SILVER_EVENTS}'
    """)

    log_run('SILVER_TO_GOLD','gold.fact_listening',param_str,_t0,'SUCCESS',
            rows_read=n_read,
            rows_inserted=m['inserted'],
            rows_updated=m['updated'],
            rows_deleted=m['deleted'])
    print(f'fact_listening: +{m["inserted"]:,} ins, ~{m["updated"]:,} upd, -{m["deleted"]:,} del')
    print(f'pipeline_state updated to version {cur_ver}')

except Exception as _e:
    log_run('SILVER_TO_GOLD','gold.fact_listening',param_str,_t0,'FAILURE',
            rows_read=n_read,error_message=str(_e))
    raise


## Proof: Fact table row count and sample

In [ ]:
total = spark.table(GOLD['fact_listening']).count()
print(f'gold.fact_listening total rows: {total:,}')

# Verify no orphan FKs (basic check)
orphan_devices = spark.sql(f"""
    SELECT COUNT(*) AS n FROM {table_ref(GOLD['fact_listening'])} f
    WHERE NOT EXISTS (
        SELECT 1 FROM {table_ref(GOLD['dim_device'])} d WHERE d.device_sk=f.device_sk)
""").first()['n']
print(f'Orphan device FKs: {orphan_devices} (expect 0)')

display(spark.sql(f"""
    SELECT data_source, COUNT(*) AS plays, ROUND(SUM(minutes_played),1) AS minutes
    FROM {table_ref(GOLD['fact_listening'])}
    GROUP BY data_source ORDER BY plays DESC
"""))
